In [ ]:
import globus_sdk

# Replace with your registered Globus App Client ID
CLIENT_ID = "4a8b00a7-c6e3-4ce3-b443-94bf8b75db9d"
auth_client = globus_sdk.NativeAppAuthClient(CLIENT_ID)

# Request access to the Transfer API
auth_client.oauth2_start_flow(requested_scopes=globus_sdk.TransferClient.scopes.all)
authorize_url = auth_client.oauth2_get_authorize_url()

print(f"Click this link, log in, and copy the authorization code:\n{authorize_url}")

In [ ]:
auth_code = input("Paste your authorization code here: ").strip()
token_response = auth_client.oauth2_exchange_code_for_tokens(auth_code)

# Extract the transfer token and create the TransferClient
transfer_token = token_response.by_resource_server["transfer.api.globus.org"]["access_token"]
authorizer = globus_sdk.AccessTokenAuthorizer(transfer_token)
tc = globus_sdk.TransferClient(authorizer=authorizer)

print("TransferClient initialized successfully.")

In [ ]:
# UCSD
SOURCE_ENDPOINT_ID = "0b302e0b-cefe-4e67-a5f1-9e7c9395d006"
SOURCE_PATH = "/cylon/web/panoseti-palomar/DATA/L0/ "

# List the contents of the directory
try:
    ls_response = tc.operation_ls(SOURCE_ENDPOINT_ID, path=SOURCE_PATH)
    print(f"Contents of {SOURCE_PATH}:")
    for item in ls_response:
        # Prints the item type (file/dir) and its name
        print(f"[{item['type']}] {item['name']}")
except globus_sdk.TransferAPIError as e:
    print(f"Error accessing endpoint: {e.message}")

In [ ]:
DEST_ENDPOINT_ID = "f57f0ebd-5311-11f1-94cb-02535127e3d7"
DEST_PATH = "/mnt/beegfs/pal-data"

# 1. Initialize the TransferData object
tdata = globus_sdk.TransferData(
    source_endpoint=SOURCE_ENDPOINT_ID,
    destination_endpoint=DEST_ENDPOINT_ID,
    label="Jupyter Filtered Recursive Transfer",
)

# 2. Define the filter rules (order matters!)
# Items that match the first rule apply. If it matches 'exclude', it gets dropped.
tdata["filter_rules"] = [
    {
        "DATA_TYPE": "filter_rule",
        "method": "include",
        "type": "file",  # Only apply this rule to files
        "name": "*.txt",  # Shell globbing is supported
    },
    {
        "DATA_TYPE": "filter_rule",
        "method": "exclude",
        "type": "file",
        "name": "*",  # Exclude all other files not matched by the previous rule
    },
]

# 3. Add the item to transfer and set recursive to True
tdata.add_item(source_path=SOURCE_PATH, destination_path=DEST_PATH, recursive=True)

# 4. Submit the task
transfer_result = tc.submit_transfer(tdata)

task_id = transfer_result["task_id"]
print("Transfer task submitted successfully!")
print(f"Task ID: {task_id}")